# PySpark 6: structured streaming

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-06-structured-streaming.ipynb)

The same event can arrive in two files. Your job must publish it once within the deduplication window and pick up where it left off after a restart.

Run a file stream with one file per micro-batch. Trace three separate concerns: event time controls late-data handling, the checkpoint records progress and state, and the sink determines what happens if a write is retried. Then run an event-time aggregation and compare its output with the raw event timestamps.

**Check your work:** inspect the actual sink rows after processing both files, then restart with the same checkpoint and confirm no new batch runs.

## Setup and sample data

This module runs on the same four tables as PySpark 1, so each module can run on its own. The setup cell installs PySpark when it is missing and starts a local session.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {"pyspark": "pyspark>=4.0,<4.1"}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

In [ ]:
import json

import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

# Python workers must use this notebook's interpreter, and every timestamp in the lessons is UTC.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["TZ"] = "UTC"
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-lessons-")

spark = (
    SparkSession.builder
    .appName("pyspark-lessons")
    .master("local[2]")  # driver and executor share one JVM: the development topology
    .config("spark.sql.shuffle.partitions", "4")  # 200 is the default; tiny data needs few
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "| shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

from pyspark.sql import functions as F

## Inputs for this notebook

Run the setup cells to start a local Spark session. The examples land their own JSON files and use local checkpoint directories.

## Process arriving files with a checkpoint

Structured Streaming lets you express streaming logic with DataFrame APIs. Spark runs the query continuously or in micro-batches.

The cell below lands two JSON files in a directory (the second repeats an event), streams them
with a watermark and `dropDuplicatesWithinWatermark`, and writes each micro-batch to a Parquet
table with `foreachBatch`. The source processes one file per micro-batch, so the duplicate
in the second file tests deduplication across batches. Each batch overwrites its own `batch_id=` folder, so a batch that Spark
retries after a failure replaces its earlier attempt instead of adding duplicate rows. The batch stays a
distributed DataFrame: never `collect()` it to the driver inside `foreachBatch`. `trigger(availableNow=True)` processes everything available
and stops - the scheduled-batch pattern. On Databricks the source would be Auto Loader
(`format("cloudFiles")`) and the sink a Delta `MERGE`.

In [ ]:
demo_dir = tempfile.mkdtemp(prefix="stream-dedup-", dir=WORK_DIR)
landing_dir = os.path.join(demo_dir, "landing", "events")
checkpoint_dir = os.path.join(demo_dir, "checkpoints", "events_silver")
os.makedirs(landing_dir, exist_ok=True)
landed = [
    [("e1", "u1", "2026-01-06T09:00:00", "view"), ("e2", "u1", "2026-01-06T09:02:00", "add_to_cart")],
    [("e2", "u1", "2026-01-06T09:02:00", "add_to_cart"), ("e3", "u2", "2026-01-06T09:05:00", "purchase")],
]
columns = ["event_id", "user_id", "event_ts", "event_type"]
for index, batch in enumerate(landed):
    with open(os.path.join(landing_dir, f"part-{index}.json"), "w") as handle:
        handle.write("\n".join(json.dumps(dict(zip(columns, row))) for row in batch))

silver_dir = os.path.join(demo_dir, "silver", "events")
batch_ids = []


def merge_batch(batch_df, batch_id):
    batch_ids.append(batch_id)
    # Idempotent per batch: a retried batch overwrites its own folder instead of appending twice.
    batch_df.write.mode("overwrite").parquet(os.path.join(silver_dir, f"batch_id={batch_id}"))


def silver_event_ids():
    silver = spark.read.parquet(silver_dir)
    return sorted(row.event_id for row in silver.select("event_id").collect())  # tiny result


def run_stream():
    query = (
        spark.readStream
        .schema("event_id string, user_id string, event_ts timestamp, event_type string")
        .option("maxFilesPerTrigger", 1)
        .json(landing_dir)
        .withWatermark("event_ts", "30 minutes")
        .dropDuplicatesWithinWatermark(["event_id"])
        .writeStream
        .foreachBatch(merge_batch)
        .option("checkpointLocation", checkpoint_dir)
        .trigger(availableNow=True)
        .start()
    )
    try:
        assert query.awaitTermination(60), "stream did not terminate"
    finally:
        query.stop()


run_stream()
print("silver events:", silver_event_ids(), "batches:", batch_ids)
assert silver_event_ids() == ["e1", "e2", "e3"]

batches_before = len(batch_ids)
run_stream()  # same checkpoint: the files are already committed, so nothing is reprocessed
assert len(batch_ids) == batches_before
print("second run read nothing new - the checkpoint remembers progress")

Real-world points:

+ Checkpoint locations are required for fault tolerance.
+ Watermarks define how late data can arrive before Spark drops state.
+ Deduplication needs a stable event key.
+ Streaming writes must be idempotent or carefully checkpointed.

**How it works.** A streaming query is a batch plan re-executed on every trigger. State for deduplication and aggregations lives in the state store under `checkpointLocation`, and a watermark bounds how long that state is kept. The driver keeps the query running and schedules one micro-batch job per trigger; the sink commits each batch transactionally.

**Example.** With a 30-minute watermark and a maximum observed event time of 10:40, the watermark is approximately 10:10. In append mode, Spark can finalize the 10:00–10:05 window once the watermark passes its end. State size also depends on the number of keys and how far event time advances.

**Watch out for.** Complete mode emits the full aggregate result on each trigger and retains old aggregate state. For a windowed aggregation, use a watermark and a compatible output mode when you need Spark to evict old state.

## Keep and expire custom per-key state

Built-in stateful operators (windowed aggregation, `dropDuplicatesWithinWatermark`, stream-stream joins) cover most pipelines. When the rule is your own - a session that closes after 30 idle minutes, a running balance, "alert on the third failed payment" - Spark 4 adds `transformWithState`, called `transformWithStateInPandas` in Python. It replaces `applyInPandasWithState` (and `mapGroupsWithState` / `flatMapGroupsWithState` in Scala) as the arbitrary-state API.

You write a `StatefulProcessor`. `init` declares named state variables - `ValueState`, `ListState` (cheap appends) or `MapState` (point lookups), each with an optional TTL that evicts expired state for you. `handleInputRows` runs once per key per micro-batch, and timers (processing or event time) fire `handleExpiredTimer`:

```python
from typing import Iterator

import pandas as pd
from pyspark.sql.streaming import StatefulProcessor, StatefulProcessorHandle

spark.conf.set(
    "spark.sql.streaming.stateStore.providerClass",
    "org.apache.spark.sql.execution.streaming.state.RocksDBStateStoreProvider",
)


class EventCounter(StatefulProcessor):
    def init(self, handle: StatefulProcessorHandle) -> None:
        self.seen = handle.getValueState("seen", "n long")  # one value per user_id

    def handleInputRows(self, key, rows, timerValues) -> Iterator[pd.DataFrame]:
        total = self.seen.get()[0] if self.seen.exists() else 0
        for batch in rows:  # pandas DataFrames holding this key's new rows
            total += len(batch)
        self.seen.update((total,))
        yield pd.DataFrame({"user_id": [key[0]], "events_seen": [total]})

    def close(self) -> None:
        pass


counts = (
    events_stream.groupBy("user_id")
    .transformWithStateInPandas(
        statefulProcessor=EventCounter(),
        outputStructType="user_id string, events_seen long",
        outputMode="Update",
        timeMode="None",  # "ProcessingTime" or "EventTime" to register timers
    )
)
```

The state lives in the checkpoint's state store, so the same checkpoint rules apply: one checkpoint per query, never shared. The PySpark API reference example sets the RocksDB state store provider as above, and the Python worker needs `pandas`, `pyarrow` and `protobuf` installed (the `pyspark[connect]` extra brings `protobuf`).

**Watch out for.** keeping per-key state with no TTL or timer. A key that never comes back holds its state forever, and the state store grows until checkpoints slow down. Give every state variable a TTL or clear it from a timer.

## Recover a streaming query after a failed write

A **checkpoint** records source progress and state so Spark can resume the same query after a restart. A `foreachBatch` callback can receive the same batch again if the sink write succeeds but Spark does not record the batch as complete. The sink must handle that retry.

**Where it matters.** Use a stable business key and source sequence for a merge, or record the batch ID in the same transaction as the output. A batch ID is scoped to the query's checkpoint history.

**Watch out for.** Give each query a durable checkpoint path of its own. Starting with a new path starts a new history. Before changing the source, stateful operators, or state schema, check which changes can reuse the existing checkpoint.

### Example

Batch 12 writes orders 101 and 102, then the driver exits before committing progress. On restart with the same checkpoint, Spark can pass batch 12 to the callback again. The earlier Parquet example overwrites the same batch directory. A transactional production sink can merge by order ID and source version. In either case, verify that the result contains two orders after the restart.

## Check yourself

The lessons above end with takeaways. Re-run each cell in order, and use `explain` on any query you do not understand before you move on.

## Key takeaways

- Checkpoints make a streaming query restartable. Keep one per query, and never share it.
- A watermark bounds how late data can arrive and how much state Spark keeps.
- On Databricks, use Auto Loader and a Delta MERGE instead of this local pattern.
- For custom per-key rules, use `transformWithStateInPandas` with a TTL or timer on every state variable.

**Next:** [PySpark 7: monitoring and performance](pyspark-07-monitoring-and-performance.ipynb)

## Run a file stream with event-time windows

A JSON directory is a real Structured Streaming source. Each file is placed before the bounded `availableNow` trigger starts. A five-minute tumbling window groups events using event time, not ingestion time. Complete mode exposes the current aggregate state, so this test does not assert append-mode watermark eviction. Expect two windows with counts 2 and 1. The query always stops and its temporary checkpoint is removed.

[Spark 4.0 streaming guide](https://spark.apache.org/docs/4.0.0/streaming/apis-on-dataframes-and-datasets.html)

In [ ]:
import json
import tempfile
import uuid
from pathlib import Path
from pyspark.sql import functions as F

with tempfile.TemporaryDirectory(prefix='stream-window-') as directory:
    root = Path(directory)
    source = root / 'input'
    source.mkdir()
    events = [{'ts':'2026-01-01 10:01:00','amount':10},
              {'ts':'2026-01-01 10:02:00','amount':20},
              {'ts':'2026-01-01 10:07:00','amount':5}]
    (source/'batch.json').write_text('\n'.join(json.dumps(r) for r in events))
    stream = spark.readStream.schema('ts timestamp, amount int').json(str(source))
    totals = stream.withWatermark('ts','10 minutes').groupBy(F.window('ts','5 minutes')).agg(F.count('*').alias('events'),F.sum('amount').alias('amount'))
    name = 'windows_' + uuid.uuid4().hex
    query = (totals.writeStream.format('memory').queryName(name).outputMode('complete')
             .option('checkpointLocation', str(root/'checkpoint')).trigger(availableNow=True).start())
    try:
        assert query.awaitTermination(60), 'stream did not terminate'
        result = spark.table(name).orderBy('window.start')
        assert [(r.events,r.amount) for r in result.collect()] == [(2,30),(1,5)]
        result.show(truncate=False)
    finally:
        query.stop()
        spark.catalog.dropTempView(name)